#### Import Library

In [14]:
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader
from dataclasses import dataclass
from datasets import load_dataset, concatenate_datasets, ClassLabel, Dataset
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, AutoTokenizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
import torch
import evaluate
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import warnings
warnings.filterwarnings('ignore')
import random
import os
from nltk import word_tokenize 
from nltk.corpus import stopwords
from collections import Counter

nltk.download("wordnet")
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\prk\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\prk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\prk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\prk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [15]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

#### Process CSV Datasets

In [16]:
df = pd.read_csv("../Datasets/data.csv")
df['label'] = df['Feeling']
df['text'] = df['Tweets']

df.drop(columns=['Tweets', 'Sl no', 'Search key', 'Feeling'], inplace=True)
df = df.sample(frac=1).reset_index(drop=True)

df.to_csv("../Datasets/process_data.csv")

In [17]:
df_indo = pd.read_csv('../Datasets/PRDECT-ID Dataset.csv')

drop_columns = ['Category', 'Product Name', 'Location', 'Price', 'Overall Rating', 'Number Sold', 'Total Review', 'Customer Rating', 'Sentiment', 'Customer Review', 'Emotion']

df_indo['text'] = df_indo['Customer Review']
df_indo['label'] = df_indo['Emotion']
df_indo.drop(columns=drop_columns, inplace=True)

df_indo = df_indo.sample(frac=1).reset_index(drop=True)
df_indo.to_csv("../Datasets/indo-data-review.csv")

#### Script for loading, processing data and training.

In [18]:
DISTIL_BERT = "distilbert-base-uncased"

MODELS = [
    "distilbert-base-uncased",
    "roberta-base",
    "bert-base-uncased",
    "xlnet-base-cased"
]

DATASETS_LINKS: list[str] = [
    {
        'name': 'dair-ai/emotion',
        'hf_dataset': True
    },
    {
        'name': '../Datasets/indo-data-review.csv',
        'hf_dataset': False
    },
    {
        'name': '../Datasets/process_data.csv',
        'hf_dataset': False
    },
]

In [19]:
@dataclass
class DataLoaderSettings:
    '''
    Settings or attributes that is going to be past in the data loader class contains information for dataset link, keys from the dataset (train, test, validation), text_col and label_col of the dataset
    '''
    dataset_link: str
    keys: list[str]
    text_col: str
    label_col: str
    with_augmentation: bool
    hf_dataset: bool = True

@dataclass
class TrainingInformation:
    pretrained_model: str
    epoch: int
    dataset_name: str

@dataclass
class DatasetSettings:
    label_col: str
    text_col: str
    tokenizer_link: str

In [20]:
# Easy data augmentation techniques for text classification
# Jason Wei and Kai Zou
from nltk.corpus import wordnet
import random
import re
from random import shuffle
random.seed(1)

#stop words list
stop_words = ['i', 'me', 'my', 'myself', 'we', 'our', 
			'ours', 'ourselves', 'you', 'your', 'yours', 
			'yourself', 'yourselves', 'he', 'him', 'his', 
			'himself', 'she', 'her', 'hers', 'herself', 
			'it', 'its', 'itself', 'they', 'them', 'their', 
			'theirs', 'themselves', 'what', 'which', 'who', 
			'whom', 'this', 'that', 'these', 'those', 'am', 
			'is', 'are', 'was', 'were', 'be', 'been', 'being', 
			'have', 'has', 'had', 'having', 'do', 'does', 'did',
			'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or',
			'because', 'as', 'until', 'while', 'of', 'at', 
			'by', 'for', 'with', 'about', 'against', 'between',
			'into', 'through', 'during', 'before', 'after', 
			'above', 'below', 'to', 'from', 'up', 'down', 'in',
			'out', 'on', 'off', 'over', 'under', 'again', 
			'further', 'then', 'once', 'here', 'there', 'when', 
			'where', 'why', 'how', 'all', 'any', 'both', 'each', 
			'few', 'more', 'most', 'other', 'some', 'such', 'no', 
			'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 
			'very', 's', 't', 'can', 'will', 'just', 'don', 
			'should', 'now', '']

#cleaning up text
def get_only_chars(line):

    clean_line = ""

    line = line.replace("’", "")
    line = line.replace("'", "")
    line = line.replace("-", " ") #replace hyphens with spaces
    line = line.replace("\t", " ")
    line = line.replace("\n", " ")
    line = line.lower()

    for char in line:
        if char in 'qwertyuiopasdfghjklzxcvbnm ':
            clean_line += char
        else:
            clean_line += ' '

    clean_line = re.sub(' +',' ',clean_line) #delete extra spaces
    if clean_line[0] == ' ':
        clean_line = clean_line[1:]
    return clean_line

########################################################################
# Synonym replacement
# Replace n words in the sentence with synonyms from wordnet
########################################################################

#for the first time you use wordnet
#import nltk
#nltk.download('wordnet')

def synonym_replacement(words, n):
	new_words = words.copy()
	random_word_list = list(set([word for word in words if word not in stop_words]))
	random.shuffle(random_word_list)
	num_replaced = 0
	for random_word in random_word_list:
		synonyms = get_synonyms(random_word)
		if len(synonyms) >= 1:
			synonym = random.choice(list(synonyms))
			new_words = [synonym if word == random_word else word for word in new_words]
			#print("replaced", random_word, "with", synonym)
			num_replaced += 1
		if num_replaced >= n: #only replace up to n words
			break

	#this is stupid but we need it, trust me
	sentence = ' '.join(new_words)
	new_words = sentence.split(' ')

	return new_words

def get_synonyms(word):
	synonyms = set()
	for syn in wordnet.synsets(word): 
		for l in syn.lemmas(): 
			synonym = l.name().replace("_", " ").replace("-", " ").lower()
			synonym = "".join([char for char in synonym if char in ' qwertyuiopasdfghjklzxcvbnm'])
			synonyms.add(synonym) 
	if word in synonyms:
		synonyms.remove(word)
	return list(synonyms)

########################################################################
# Random deletion
# Randomly delete words from the sentence with probability p
########################################################################

def random_deletion(words, p):
	#obviously, if there's only one word, don't delete it
    if len(words) == 1:
        return words

	#randomly delete words with probability p
    new_words = []
    for word in words:
        r = random.uniform(0, 1)
        if r > p:
            new_words.append(word)

	#if you end up deleting all words, just return a random word
    if len(new_words) == 0:
        rand_int = random.randint(0, len(words)-1)
        return [words[rand_int]]

    return new_words

########################################################################
# Random swap
# Randomly swap two words in the sentence n times
########################################################################

def random_swap(words, n):
	new_words = words.copy()
	for _ in range(n):
		new_words = swap_word(new_words)
	return new_words

def swap_word(new_words):
	random_idx_1 = random.randint(0, len(new_words)-1)
	random_idx_2 = random_idx_1
	counter = 0
	while random_idx_2 == random_idx_1:
		random_idx_2 = random.randint(0, len(new_words)-1)
		counter += 1
		if counter > 3:
			return new_words
	new_words[random_idx_1], new_words[random_idx_2] = new_words[random_idx_2], new_words[random_idx_1] 
	return new_words

########################################################################
# Random insertion
# Randomly insert n words into the sentence
########################################################################

def random_insertion(words, n):
	new_words = words.copy()
	for _ in range(n):
		add_word(new_words)
	return new_words

def add_word(new_words):
	synonyms = []
	counter = 0
	while len(synonyms) < 1:
		random_word = new_words[random.randint(0, len(new_words)-1)]
		synonyms = get_synonyms(random_word)
		counter += 1
		if counter >= 10:
			return
	random_synonym = synonyms[0]
	random_idx = random.randint(0, len(new_words)-1)
	new_words.insert(random_idx, random_synonym)

########################################################################
# main data augmentation function
########################################################################

def eda(sentence, alpha_sr=0.1, alpha_ri=0.1, alpha_rs=0.1, p_rd=0.1, num_aug=9):
	
	sentence = get_only_chars(sentence)
	words = sentence.split(' ')
	words = [word for word in words if word is not '']
	num_words = len(words)
	
	augmented_sentences = []
	num_new_per_technique = int(num_aug/4)+1

	#sr
	if (alpha_sr > 0):
		n_sr = max(1, int(alpha_sr*num_words))
		for _ in range(num_new_per_technique):
			a_words = synonym_replacement(words, n_sr)
			augmented_sentences.append(' '.join(a_words))

	#ri
	if (alpha_ri > 0):
		n_ri = max(1, int(alpha_ri*num_words))
		for _ in range(num_new_per_technique):
			a_words = random_insertion(words, n_ri)
			augmented_sentences.append(' '.join(a_words))

	#rs
	if (alpha_rs > 0):
		n_rs = max(1, int(alpha_rs*num_words))
		for _ in range(num_new_per_technique):
			a_words = random_swap(words, n_rs)
			augmented_sentences.append(' '.join(a_words))

	#rd
	if (p_rd > 0):
		for _ in range(num_new_per_technique):
			a_words = random_deletion(words, p_rd)
			augmented_sentences.append(' '.join(a_words))

	augmented_sentences = [get_only_chars(sentence) for sentence in augmented_sentences]
	shuffle(augmented_sentences)

	#trim so that we have the desired number of augmented sentences
	if num_aug >= 1:
		augmented_sentences = augmented_sentences[:num_aug]
	else:
		keep_prob = num_aug / len(augmented_sentences)
		augmented_sentences = [s for s in augmented_sentences if random.uniform(0, 1) < keep_prob]

	#append the original sentence
	augmented_sentences.append(sentence)

	return augmented_sentences

In [21]:
class DataProcessor:
    '''
    Separate Class that works with data loader for processing the data (text processing, text augmentation and other important part of the data loading process)
    '''

    def __init__(self, save_path='../Datasets/'):
        self.stop_words = stopwords.words("english")
        self.save_path = save_path
    
    def balance_dataset(self, dataset: Dataset, dataset_name: str, with_augmentation: bool, text_col='text', label_col='label'):
        '''
        Balance dataset by randomly discarding data or augmenting data.

        Args:
        - dataset: Dataset, the dataset to balance.
        - dataset_name: str, name of the dataset file.
        - with_augmentation: bool, whether to use augmentation for balancing.
        - text_col: str, column name containing text data.
        - label_col: str, column name containing labels.

        Returns:
        - Dataset, balanced dataset.
        '''
        augmented_desc = "_augmented" if with_augmentation else ""
        dataset_name += '.csv'
        file_name = self.save_path + augmented_desc + dataset_name
        if os.path.exists(file_name) and with_augmentation:
            df = pd.read_csv(self.save_path + dataset_name)
            return Dataset.from_pandas(df)

        # Count occurrences of each label
        label_counts = Counter(dataset[label_col])
        max_count = max(label_counts.values())
        min_count = min(label_counts.values())  # Get the smallest label count

        balanced_data = []
        for label, count in label_counts.items():
            # Extract rows with the current label
            subset = [row for row in dataset if row[label_col] == label]

            if with_augmentation:
                # Augment the data to match the max count
                while len(subset) < max_count:
                    sample = random.choice(subset)
                    augmented_texts = eda(sample[text_col])  # Apply augmentation
                    for aug_text in augmented_texts:
                        if len(subset) < max_count:
                            augmented_sample = sample.copy()
                            augmented_sample[text_col] = aug_text
                            subset.append(augmented_sample)
                        else:
                            break
            else:
                # Randomly downsample if necessary
                if count > min_count:  
                    subset = random.sample(subset, min_count)

            balanced_data.extend(subset)

        df_balanced = pd.DataFrame(balanced_data)
        df_balanced.to_csv(file_name, index=False)

        return Dataset.from_list(balanced_data)


    def convert_labels_to_classlabel(self, dataset: Dataset, label_col: str) -> Dataset:
        '''
        Convert labels from a dataset into Class Label
        
        Args:
        - dataset: Dataset, dataset that is going to be converted into class label
        - label_col: str, name of the column that is going to be changed

        Returns:
        - dataset: Dataset, dataset that have been processed
        '''
        unique_labels = list(set(dataset[label_col]))
        class_label_feature = ClassLabel\
            (num_classes=len(unique_labels), names=[str(label) for label in unique_labels])

        dataset = dataset\
            .map(lambda example: {label_col: class_label_feature.str2int(str(example[label_col]))})
        dataset = dataset.cast_column(label_col, class_label_feature)

        return dataset

    def process_text(self, dataset: Dataset, text_col: str) -> Dataset:
        '''
        Process text column from the dataset, actions: Remove Stop Words, and non alphabetic words
        
        Args:
        - dataset: Dataset, dataset that is going to be processed
        - text_col: str, name of the text column
        '''
        def process_text(sample) -> str:
            text = sample[text_col]
            words = self.tokenize(text)
            words = self.remove_stopwords(words)

            sample[text_col] = ' '.join(words)
            return sample

        dataset = dataset.map(process_text)
        return dataset

    def tokenize(self, text: str) -> list[str]:
        '''
        Tokenize text into words
        
        Args:
        - text: str, text that is going to be tokenize
        
        Returns:
        - list[str], result of the tokenize
        '''

        words = word_tokenize(text)
        return words

    def remove_stopwords(self, words: list[str]) -> list[str]:
        '''
        Remove words that is in the list of stop words and not alphabetic

        Args:
        - words: list[str]
        
        Returns:
        - list[str]
        '''

        words = [word for word in words if word not in self.stop_words and word.isalpha()]
        return words


In [22]:
class DataLoader:
    '''
    Class for load data from hugging or csv file
    '''
    def __init__(self, loader_settings: DataLoaderSettings, save_path='../Experiments/Datasets/'):
        self.save_path = save_path
        self.settings = loader_settings
        self.file_name = self.settings.dataset_link.replace("/", " ")
        self.data_processor = DataProcessor()
        self.loaded = self.load_dataset()
        self.processed = self.process_dataset(self.settings.with_augmentation)
        
        print(f'Loaded: {self.loaded}, Processed: {self.processed}')
        
    def process_dataset(self, with_augmentation=True) -> bool:
        '''
        Process dataset using the data_processor class
        '''
        if not hasattr(self, 'dataset'):
            raise AttributeError("Dataset has not been loaded")
        print(self.dataset.features[self.settings.label_col])
        self.class_names = self.dataset.features[self.settings.label_col].names
        
        self.dataset = self.data_processor.process_text\
            (dataset=self.dataset, text_col=self.settings.text_col)
            
        self.dataset = self.data_processor.balance_dataset\
            (dataset=self.dataset, dataset_name=self.settings.dataset_link.replace('/', ' '), with_augmentation=with_augmentation, text_col='text')
        
        self.dataset = self.data_processor.convert_labels_to_classlabel\
            (dataset=self.dataset, label_col=self.settings.label_col)
        return True

    def load_dataset(self) -> bool:
        '''
        Load Dataset from a csv file or hugging face dataset
        '''
        if not self.settings.hf_dataset:
            try:
                df = pd.read_csv(self.settings.dataset_link)
                
                labels = list(df['label'].unique())
                class_label = ClassLabel(names=labels)
                df['label'] = df['label'].apply(lambda x: labels.index(x))
                
                self.dataset = Dataset.from_pandas(df)
                self.dataset = self.dataset.cast_column("label", class_label)
                return True
            except Exception as e:
                print(f"Loading dataset went wrong: {e}")
                return False
        try:
            dataset = load_dataset(self.settings.dataset_link, trust_remote_code=True)
            merged_dataset = None

            if not isinstance(dataset, dict):
                self.dataset = dataset
                return True

            for key in self.settings.keys:
                if key not in dataset:
                    continue

                dataset_partition = dataset[key]
                
                if merged_dataset == None:
                    merged_dataset = dataset_partition
                else:
                    merged_dataset = concatenate_datasets([merged_dataset, dataset_partition])
            
            self.dataset = merged_dataset
            return True
        except Exception as e:
            print(f'Loading {self.settings.dataset_link} Error: {e}')
            return False

In [23]:

class CustomDataset:
    """
    Custom dataset class for tokenizing text data.

    Attributes:
    - dataset: DataFrame loaded from CSV
    - tokenizer: Tokenizer for text processing
    - max_length: Maximum token length
    """
    
    def __init__(self, dataset_settings: DatasetSettings, data_loader: DataLoader, max_length=512):
        self.tokenizer = AutoTokenizer.from_pretrained(dataset_settings.tokenizer_link)
        self.settings = dataset_settings
        self.data_loader = data_loader
        self.max_length = max_length
        self.label_encoder = LabelEncoder()
        self.splitted = self.split_dataset()
        self.tokenized = self.tokenize_datasets()
        print(f'Splitted: {self.splitted}')
        print(f'Tokenized: {self.tokenized}')
    
    def count_unique_labels(self) -> int:
        """
        Counts the number of unique labels in the dataset.

        Returns:
        - int: The number of unique labels.
        """
        try:
            label_column = self.settings.label_col

            # Get unique labels from dataset
            unique_labels = set(self.train[label_column])

            print(f"Number of unique labels: {len(unique_labels)}")
            return len(unique_labels)

        except Exception as e:
            print(f"Error counting unique labels: {e}")
            return 0

    def __len__(self):
        return len(self.data)
    
    def get_class_names(self):
        return self.data_loader.class_names

    def split_dataset(self, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42) -> bool:
        """
        Split the dataset into training, validation, and testing sets.

        Args:
        - train_ratio (float): Proportion of the dataset to use for training.
        - val_ratio (float): Proportion of the dataset to use for validation.
        - test_ratio (float): Proportion of the dataset to use for testing.
        - random_state (int): Seed for reproducibility.

        Returns:
        - bool: True if the split was successful, False otherwise.
        """
        if not hasattr(self, "data_loader"):
            print("Dataset is not loaded.")
            return False

        if not (0 < train_ratio < 1 and 0 < val_ratio < 1 and 0 < test_ratio < 1 and train_ratio + val_ratio + test_ratio == 1):
            print("Invalid split ratios. Ensure they sum to 1.")
            return False

        dataset = self.data_loader.dataset
        
        try:
            # First, split into train and temp (val + test)
            train_test_split = dataset.train_test_split(test_size=(1 - train_ratio), seed=seed, stratify_by_column=self.settings.label_col)
            train_data = train_test_split["train"]
            temp_data = train_test_split["test"]

            # Compute relative validation split
            val_size = val_ratio / (val_ratio + test_ratio)  # Normalize val/test split
            val_test_split = temp_data.train_test_split(test_size=(1 - val_size), seed=seed, stratify_by_column=self.settings.label_col)

            self.train = train_data
            self.val = val_test_split["train"]
            self.test = val_test_split["test"]

            print(f"Dataset split complete: Train({len(self.train)}), Val({len(self.val)}), Test({len(self.test)})")
            return True

        except Exception as e:
            print(f"Error splitting dataset: {e}")
            return False

    def tokenize_datasets(self):
        """
        Tokenizes the train, validation, and test datasets using the tokenizer.

        This function modifies self.train, self.val, and self.test in-place.
        """
        if not hasattr(self, "train") or self.train is None:
            print("Training dataset is not loaded.")
            return False
        if not hasattr(self, "val") or self.val is None:
            print("Validation dataset is not loaded.")
            return False
        if not hasattr(self, "test") or self.test is None:
            print("Testing dataset is not loaded.")
            return False

        try:
            text_column = self.settings.text_col
            label_column = self.settings.label_col
            
            # Tokenization function
            def tokenize_function(example):
                encoding = self.tokenizer(
                    example[text_column],
                    padding="max_length",
                    truncation=True,
                    max_length=self.max_length
                )
                encoding["labels"] = [torch.tensor(label, dtype=torch.long) for label in example[label_column]]
                return encoding

            self.train = self.train.map(tokenize_function, batched=True)
            self.val = self.val.map(tokenize_function, batched=True)
            self.test = self.test.map(tokenize_function, batched=True)

            print("Tokenization complete for train, val, and test datasets.")
            return True

        except Exception as e:
            print(f"Error tokenizing datasets: {e}")
            return False

In [24]:
def print_label_counts(dataset: Dataset, label_col='label'):
    '''
    Prints the count of each label in the dataset.

    Args:
    - dataset: Dataset, the dataset to analyze.
    - label_col: str, column name containing labels.
    '''
    label_counts = Counter(dataset[label_col])
    print("Label counts:")
    for label, count in label_counts.items():
        print(f"{label}: {count}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels, average="macro")
    recall = recall_metric.compute(predictions=predictions, references=labels, average="macro")
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"]
    }

def save_training_result(results: dict[str, float], training_information: "TrainingInformation", dataset_augmented: bool) -> str:
    '''
    Save results from evaluation after training using pretrained model.
    
    Args:
    - results: dictionary of the evaluation results (acc, precision, and other metrics)
    - training_information: training configuration that is going to be used for the file name.
    
    Returns:
    - bool: If the saving is successful or not.
    '''
    augmented_string = "_Augmented" if dataset_augmented else "" 
    try:
        folder_name = f"results_{training_information.pretrained_model}_epoch{training_information.epoch}{augmented_string}"
        folder_name = folder_name.replace("/", "_").replace(" ", "_")
        os.makedirs(folder_name, exist_ok=True)
     
        filename = os.path.join(folder_name, "results.json")
        
        with open(filename, "w") as f:
            json.dump(results, f, indent=4)
        
        print(f"Results saved to {filename}")
        return folder_name 
    except Exception as e:
        print(f'Error saving result: {e}')
        return ""

def train_model(dataset: CustomDataset, training_information: TrainingInformation):
    model = AutoModelForSequenceClassification.from_pretrained(training_information.pretrained_model, num_labels=dataset.count_unique_labels())
    print_label_counts(dataset=dataset.train, label_col=dataset.settings.label_col)
    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=training_information.epoch,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        evaluation_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="epoch",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset.train,
        eval_dataset=dataset.val,
        compute_metrics=compute_metrics
    )

    trainer.train()

    evaluation_result = trainer.evaluate(dataset.test)
    folder_path = save_training_result(evaluation_result, training_information,\
        dataset_augmented=dataset.data_loader.settings.with_augmentation)

    predictions = trainer.predict(dataset.test)
    eval_pred = (predictions.predictions, predictions.label_ids)
    
    class_names = dataset.get_class_names()
    generate_confusion_matrix(eval_pred, list(range(dataset.count_unique_labels())), os.path.join(folder_path, "confusion_matrix.jpeg"), class_names)

def generate_confusion_matrix(eval_pred, labels, save_path, class_names=None):
    """
    Generates and saves the confusion matrix as a .jpg file.

    Args:
    - eval_pred: Tuple containing (logits, labels).
    - labels: List of unique labels.
    - class_names: List of class names (optional).
    - save_path: File path to save the confusion matrix image.
    """
    logits, true_labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    cm = confusion_matrix(true_labels, predictions, labels=labels)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")

    plt.savefig(save_path, format="jpg", dpi=300)
    plt.close()
    print(f"Confusion matrix saved to {save_path}")

#### Training

In [25]:
loader_settings = DataLoaderSettings(
    dataset_link=DATASETS_LINKS[1]['name'],
    keys=['train', 'test', 'validation'],
    label_col='label',
    text_col='text',
    with_augmentation=True,
    hf_dataset=DATASETS_LINKS[1]['hf_dataset']
)

data_loader = DataLoader(loader_settings)

dataset_settings = DatasetSettings(
    label_col='label',
    text_col='text',
    tokenizer_link=DISTIL_BERT
)

dataset = CustomDataset(dataset_settings, data_loader)

training_information = TrainingInformation(
    dataset_name=DATASETS_LINKS[1]['name'],
    epoch=1,
    pretrained_model=DISTIL_BERT
)

train_model(dataset, training_information)

Casting the dataset: 100%|██████████| 5400/5400 [00:00<?, ? examples/s]


ClassLabel(names=['Anger', 'Happy', 'Fear', 'Sadness', 'Love'], id=None)


Casting the dataset: 100%|██████████| 8850/8850 [00:00<00:00, 3027699.05 examples/s]


Loaded: True, Processed: True
Dataset split complete: Train(7080), Val(885), Test(885)


Map: 100%|██████████| 885/885 [00:00<00:00, 7886.58 examples/s]


Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
Number of unique labels: 5


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Label counts:
2: 1416
0: 1416
1: 1416
4: 1416
3: 1416


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.089100,0.606932,0.774011,0.774335,0.774011,0.773997


Results saved to results_distilbert-base-uncased_epoch1_Augmented\results.json
Number of unique labels: 5
Confusion matrix saved to results_distilbert-base-uncased_epoch1_Augmented\confusion_matrix.jpeg


In [26]:
# for model in MODELS:
#     loader_settings = DataLoaderSettings(
#         dataset_link=DATASETS_LINKS[1],
#         keys=['train', 'test', 'validation'],
#         label_col='label',
#         text_col='text',
#         with_augmentation=True
#     )

#     data_loader = DataLoader(loader_settings)

#     dataset_settings = DatasetSettings(
#         label_col='label',
#         text_col='text',
#         tokenizer_link=model
#     )

#     dataset = CustomDataset(dataset_settings, data_loader)

#     training_information = TrainingInformation(
#         dataset_name=DATASETS_LINKS[1],
#         epoch=4,
#         pretrained_model=model
#     )

#     train_model(dataset, training_information)